# TRIAGE-EG Stage 1C — Qualitative Text Retrieval Evaluation

Stage 1B has verified the official OpenAI CLIP model space. Stage 1C evaluates the unchanged raw baseline only: no translation, optimization, reranking, diversification, or Recall@K claim. Human review of contact sheets is required before any retrieval-quality conclusion.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
REFRESH_REPO = os.environ.get('AIC_REFRESH_REPO', '0') == '1'
DATA_ROOT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
STAGE0_ROOT = Path(os.environ.get('AIC_STAGE0_ROOT', '/kaggle/working/triage_eg_stage0_audit'))
STAGE0_BUNDLE = os.environ.get('AIC_STAGE0_BUNDLE', '/kaggle/input/datasets/irthn1311/triage-eg-stage0-audit-bundle')
STAGE1_ROOT = Path(os.environ.get('AIC_STAGE1_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle'))
STAGE1B_ROOT = Path(os.environ.get('AIC_STAGE1B_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports'))
ASSET_ROOT = Path(os.environ.get('AIC_OPENAI_CLIP_ASSET_ROOT', '/kaggle/input/aic2026-openai-clip-vit-b32'))
OUTPUT_ROOT = Path(os.environ.get('AIC_STAGE1C_OUTPUT_ROOT', '/kaggle/working/triage_eg_stage1c_qualitative_eval'))
QUERY_SUITE_VALUE = os.environ.get('AIC_STAGE1C_QUERY_SUITE', 'configs/retrieval/stage1c_qualitative_queries.jsonl')
FRAME_TOP_K = int(os.environ.get('AIC_STAGE1C_FRAME_TOP_K', '50'))
KIS_TOP_K = int(os.environ.get('AIC_STAGE1C_KIS_TOP_K', '100'))
REVIEW_TOP_K = int(os.environ.get('AIC_STAGE1C_REVIEW_TOP_K', '10'))
DEVICE = os.environ.get('AIC_STAGE1C_DEVICE', 'auto')
BATCH_SIZE = int(os.environ.get('AIC_STAGE1C_BATCH_SIZE', '16'))
REUSE_RESULTS = os.environ.get('AIC_STAGE1C_REUSE_RESULTS', '0') == '1'
print({'ref': REPO_REF, 'data': str(DATA_ROOT), 'stage1': str(STAGE1_ROOT), 'stage1b': str(STAGE1B_ROOT), 'asset': str(ASSET_ROOT), 'output': str(OUTPUT_ROOT)})


In [ ]:
def git_result(*args, cwd=None):
    return subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True, check=False)

def git(*args, cwd=None):
    result = git_result(*args, cwd=cwd)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout')
if not (REPO_DIR / '.git').is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git('clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_DIR))
target = None
if not REFRESH_REPO:
    for candidate in (REPO_REF, f'origin/{REPO_REF}'):
        probe = git_result('rev-parse', '--verify', f'{candidate}^{{commit}}', cwd=REPO_DIR)
        if probe.returncode == 0:
            target = probe.stdout.strip()
            break
if target is None:
    git('fetch', '--depth', '1', 'origin', REPO_REF, cwd=REPO_DIR)
    target = 'FETCH_HEAD'
git('checkout', '--detach', target, cwd=REPO_DIR)
COMMIT = git('rev-parse', 'HEAD', cwd=REPO_DIR)
os.environ['AIC_RESOLVED_GIT_COMMIT'] = COMMIT
sys.path.insert(0, str(REPO_DIR / 'src'))
print('resolved commit:', COMMIT)


In [ ]:
from triage_eg.retrieval.stage1.stage0_loader import resolve_stage0_root
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1c import Stage1CConfig, preflight_stage1c
from triage_eg.retrieval.stage1c.inputs import resolve_stage1b_root

STAGE0_ROOT = resolve_stage0_root(STAGE0_ROOT, bundle_path=STAGE0_BUNDLE, search_root=Path('/kaggle/input'), excluded_roots=(DATA_ROOT,))
STAGE1_ROOT = resolve_stage1_root(STAGE1_ROOT, search_root=Path('/kaggle/input'), excluded_roots=(DATA_ROOT, STAGE0_ROOT), materialize_root=Path('/kaggle/working/triage_eg_stage1c_stage1_input'))
STAGE1B_ROOT = resolve_stage1b_root(STAGE1B_ROOT, search_root=Path('/kaggle/input'), materialize_root=Path('/kaggle/working/triage_eg_stage1c_stage1b_input'))
if not ASSET_ROOT.is_dir():
    fallback = Path('/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32')
    if fallback.is_dir():
        ASSET_ROOT = fallback
QUERY_SUITE = Path(QUERY_SUITE_VALUE)
if not QUERY_SUITE.is_absolute():
    QUERY_SUITE = REPO_DIR / QUERY_SUITE
CONFIG = Stage1CConfig(repo_root=REPO_DIR, dataset_root=DATA_ROOT, stage0_root=STAGE0_ROOT, stage1_root=STAGE1_ROOT, stage1b_root=STAGE1B_ROOT, encoder_asset_root=ASSET_ROOT, query_suite=QUERY_SUITE, output_root=OUTPUT_ROOT, frame_top_k=FRAME_TOP_K, kis_top_k=KIS_TOP_K, review_top_k=REVIEW_TOP_K, device=DEVICE, batch_size=BATCH_SIZE, overwrite=not REUSE_RESULTS, reuse_results=REUSE_RESULTS, strict_root=True, build_git_commit=COMMIT)
PREFLIGHT = preflight_stage1c(CONFIG)
print(json.dumps(PREFLIGHT, indent=2, ensure_ascii=False))


In [ ]:
import pandas as pd
from triage_eg.retrieval.stage1c import load_query_suite
queries, suite_manifest = load_query_suite(QUERY_SUITE)
display(pd.DataFrame([item.__dict__ for item in queries])[[
    'query_id', 'pair_id', 'language', 'category', 'difficulty', 'text'
]])
print('suite fingerprint:', suite_manifest['fingerprint'])


In [ ]:
selected_contract = json.loads((STAGE1B_ROOT / 'encoder/selected_encoder_contract.json').read_text())
assert selected_contract['compatibility_status'] == 'VERIFIED'
print('verified encoder:', selected_contract['selected_candidate_id'])
print('checkpoint SHA-256:', selected_contract['checkpoint_sha256'])
print('The Stage 1C runner loads this contract through the existing offline Stage 1B adapter.')


In [ ]:
print(f'Encoding plan: {len(queries)} queries in one verified-adapter batch; dimension=512')
print('Embeddings are runtime-only and are not written into the qualitative bundle.')


In [ ]:
from triage_eg.retrieval.stage1c import run_stage1c
result = run_stage1c(CONFIG)
print('Stage 1C reused:', result.reused)
print('queries completed:', result.summary['retrieval']['queries_completed'])
print('raw Top-K:', result.summary['retrieval']['raw_frame_top_k'])
print('internal KIS Top-K:', result.summary['retrieval']['kis_export_top_k'])


In [ ]:
summary = result.summary
print(json.dumps(summary['retrieval'], indent=2))
print('issues:', json.dumps(summary['issues'], indent=2))


In [ ]:
pair_rows = [json.loads(line) for line in (OUTPUT_ROOT / 'pairs/pair_diagnostics.jsonl').read_text().splitlines() if line.strip()]
display(pd.DataFrame(pair_rows)[[
    'pair_id', 'text_embedding_cosine_en_vi', 'top10_global_row_jaccard', 'top20_video_jaccard'
]])


In [ ]:
print(json.dumps(summary['structural_diagnostics'], indent=2, ensure_ascii=False))
print('Structural warnings are review heuristics; raw ranking was not changed.')


In [ ]:
from IPython.display import Image as DisplayImage, display
preview_ids = ['obj_01_en', 'action_01_en', 'obj_01_vi', 'action_01_vi', 'difficult_01_en']
for query_id in preview_ids:
    sheet = OUTPUT_ROOT / 'queries' / query_id / 'contact_sheet_top20.jpg'
    if sheet.is_file():
        print(query_id)
        display(DisplayImage(filename=str(sheet), width=800))


In [ ]:
review_csv = OUTPUT_ROOT / 'review/review_template.csv'
print('review CSV:', review_csv)
print('expected judgments:', summary['human_review']['judgments_expected'])
print('Labels remain blank for human review.')


In [ ]:
print('ENCODER =', summary['stage1b_encoder']['compatibility_status'])
print('PIPELINE =', 'WORKING' if summary['retrieval']['queries_failed'] == 0 else 'FAILED')
print('RETRIEVAL_QUALITY =', summary['retrieval_quality_status'])
assert summary['stage1b_encoder']['compatibility_status'] == 'VERIFIED'
assert summary['retrieval_quality_status'] == 'NOT_REVIEWED'


In [ ]:
from zipfile import ZipFile
from triage_eg.retrieval.stage1c import create_stage1c_bundle
zip_path = Path('/kaggle/working/triage_eg_stage1c_qualitative_eval_bundle.zip')
create_stage1c_bundle(OUTPUT_ROOT, zip_path)
with ZipFile(zip_path) as archive:
    members = archive.namelist()
assert any(name.endswith('/ranked_frames.jsonl') for name in members)
assert any(name.endswith('/contact_sheet_top20.jpg') for name in members)
assert not any(name.endswith(('.pt', '.pth', '.bin', '.npy', '.mp4', '.avi', '.mkv')) or name.startswith('logs/') for name in members)
print('DOWNLOAD ZIP:', zip_path, 'size_bytes=', zip_path.stat().st_size, 'members=', len(members))
